# Evaluation of extremes of models on 60km -> 2.2km-4x over Birmingham

In [ ]:
%reload_ext autoreload

%autoreload 2

%reload_ext dotenv
%dotenv

In [ ]:
from mlde_analysis.default_params import *

In [ ]:
import functools
import itertools
import math
import string

import cf_xarray
import IPython
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

from mlde_analysis import plot_map, STYLES, SUBREGIONS, BOX_LOCATIONS
from mlde_analysis.display import pretty_table, VAR_RANGES
from mlde_analysis.distribution import plot_freq_density_figure, compute_metrics, DIST_THRESHOLDS, stat_bias, rms, plot_freq_density
from mlde_analysis.extremes import pred_and_target_return_levels, plot_return_levels, pretty_return_levels_table
from mlde_utils import cp_model_rotated_pole, platecarree
from mlde_analysis import qq_plot, reasonable_quantiles
from mlde_analysis.utils import chained_groupby_map

In [ ]:
matplotlib.rcParams['figure.dpi'] = 300

In [ ]:
IPython.display.Markdown(desc)

In [ ]:
%reload_ext mlde_analysis.magics 
EVAL_DS, MODELS, CPM_DAS, PRED_DAS, VAR_DAS, MODELLABEL2SPEC = %load_eval_data
EVAL_DS

## Max bias

In [ ]:
for var in eval_vars:
    for normalize in [True, False]:
        max_biases = PRED_DAS[var].groupby("model").map(functools.partial(stat_bias, stat_func=xr.DataArray.max), cpm_da=VAR_DAS[var][f"target_{var}"], normalize=normalize)
        pretty_table(max_biases.groupby("model").map(rms), round=3)
        if normalize:
            style = STYLES[f"{var}Bias"]
        else:
            style = {}
        g = max_biases.plot(col="model", subplot_kws=dict(projection=cp_model_rotated_pole), **style)
        for ax in g.axs.flat:
            ax.coastlines()

## RX1day bias

In [ ]:
for var in eval_vars:    
    pred_rx1day_mean = PRED_DAS[var].groupby("dec_adjusted_year").max(dim=["time"]).mean(dim=["dec_adjusted_year", "ensemble_member"]).mean(dim=["sample_id"])
    target_rx1day_mean = VAR_DAS[var][f"target_{var}"].groupby("dec_adjusted_year").max(dim=["time"]).mean(dim=["dec_adjusted_year", "ensemble_member"])
    rx1day_bias = 100*(pred_rx1day_mean - target_rx1day_mean)/target_rx1day_mean
    style = STYLES[f"{var}Bias"] #{}
    g = rx1day_bias.plot(col="model", subplot_kws=dict(projection=cp_model_rotated_pole), **style)
    for ax in g.axs.flat:
        ax.coastlines()

## Q0.999 bias

In [ ]:
for var in eval_vars:
    for normalize in [True, False]:
        q999_biases = chained_groupby_map(PRED_DAS[var], ["model", "sample_id"], functools.partial(stat_bias, stat_func=functools.partial(xr.DataArray.quantile, q=0.999)), cpm_da=VAR_DAS[var][f"target_{var}"], normalize=normalize)
        pretty_table(q999_biases.groupby("model").map(rms), round=3)
        if normalize:
            style = STYLES[f"{var}Bias"]
        else:
            style = {}
        g = q999_biases.mean(dim="sample_id").plot(col="model", subplot_kws=dict(projection=cp_model_rotated_pole), **style)
        for ax in g.axs.flat:
            ax.coastlines()
        g = q999_biases.plot(col="model", row="sample_id", subplot_kws=dict(projection=cp_model_rotated_pole), **style)
        for ax in g.axs.flat:
            ax.coastlines()

In [ ]:
quantile_dims=["ensemble_member", "time"]

for var in eval_vars:
    IPython.display.display_markdown(f"### {var}", raw=True)
    for label, q in BOX_LOCATIONS.items():
        ds = VAR_DAS[var].cf.sel(**q, method="nearest")
        pred_da = ds[f"pred_{var}"]
        cpm_da = ds[f"target_{var}"]
        
        quantiles = reasonable_quantiles(cpm_da)
        cpm_quantiles = cpm_da.quantile(quantiles, dim=quantile_dims).rename("target_q")
    
        pred_quantiles = pred_da.quantile(quantiles, dim=quantile_dims).rename("pred_q")

        layout="constrained"

        fig, ax = plt.subplots(figsize=(5.5, 5.5), layout="constrained")

        xlabel = f"CPM \n{xr.plot.utils.label_from_attrs(da=cpm_da)}"
        ylabel = f"Predicted \n{xr.plot.utils.label_from_attrs(da=pred_da)}"

        qq_plot(ax, cpm_quantiles, pred_quantiles, title=f"Predicted quantiles vs CPM quantiles", xlabel=xlabel, ylabel=ylabel)

        plt.show()

## Individual box locations

In [ ]:
fig = plt.figure(layout="constrained", figsize=(1.5, 1.5))
ax = fig.subplots(subplot_kw={"projection": cp_model_rotated_pole})
ax.coastlines(**{"resolution": "10m", "linewidth": 0.3})
da = CPM_DAS[eval_vars[0]]
ax.set_extent((
    da.cf["X"].min(),
    da.cf["X"].max(),
    da.cf["Y"].min(),
    da.cf["Y"].max(),
))

for label, q in BOX_LOCATIONS.items():
    single_box_da = da.cf.sel(**q, method="nearest")
    ax.plot(single_box_da.cf["X"].values, single_box_da.cf["Y"].values, color='blue', markersize=0.5, marker='o', transform=cp_model_rotated_pole)
    ax.annotate(
        xy=(single_box_da.cf["X"].item(), single_box_da.cf["Y"].item()), xycoords="data",
        text=label, xytext=(0, -15), textcoords="offset pixels",
        ha='center',va="center", transform=cp_model_rotated_pole, fontsize="xx-small")
    
plt.show()

In [ ]:
da = CPM_DAS[eval_vars[0]]

rng = np.random.default_rng(42)

RANDOM_BOX_LOCATIONS = {
    srname: {
        i: { c: rng.choice(da.isel(**SUBREGIONS[srname]).cf[c], 1)[0] for c in ["X", "Y"] }
        for i in range(8) }
    for srname in SUBREGIONS.keys()
}

fig = plt.figure(layout="constrained", figsize=(1.5, 1.5))
ax = fig.subplots(subplot_kw={"projection": cp_model_rotated_pole})
ax.coastlines(**{"resolution": "10m", "linewidth": 0.3})

ax.set_extent((
    da.cf["X"].min(),
    da.cf["X"].max(),
    da.cf["Y"].min(),
    da.cf["Y"].max(),
))

for srname, sr_random_box_locations in RANDOM_BOX_LOCATIONS.items():
    for q in sr_random_box_locations.values():
        single_box_da = da.cf.sel(**q, method="nearest")
        ax.plot(single_box_da.cf["X"].values, single_box_da.cf["Y"].values, color='blue', markersize=0.5, marker='o', transform=cp_model_rotated_pole)
        
plt.show()

## Return time plots (sorting): individual grid box

In [ ]:
for var in eval_vars:
    IPython.display.display_markdown(f"### {var}", raw=True)
    rt_ds = xr.concat(
        [
            pred_and_target_return_levels(
                VAR_DAS[var].cf.sel(**q, method="nearest"), 
                var,
                n_days_per_year=360,
            ).expand_dims(location=[label])
            for label, q in BOX_LOCATIONS.items()
        ],
        dim="location",
    )

    plot_return_levels(rt_ds[f"pred_{var}_return_level"], rt_ds[f"target_{var}_return_level"], row="location")
    plt.show()

    pretty_return_levels_table(rt_ds, var)

## Return time plots (sorting): individual grid box (seasonal)

In [ ]:
for var in eval_vars:
    IPython.display.display_markdown(f"### {var}", raw=True)
    rt_ds = xr.concat(
        [
            pred_and_target_return_levels(
                VAR_DAS[var].cf.sel(**q, method="nearest").sel(time=(VAR_DAS[var]["time"]["time.season"] == season)),
                var,
                n_days_per_year=90
            ).expand_dims(location=[f"{label} {season}"])
            for (label, q), season in itertools.product(BOX_LOCATIONS.items(), ["DJF", "JJA"])
        ],
        dim="location",
    )

    plot_return_levels(rt_ds[f"pred_{var}_return_level"], rt_ds[f"target_{var}_return_level"], row="location")
    plt.show()

    pretty_return_levels_table(rt_ds, var)

## Return time plots (sorting): random individual grid box (seasonal)

In [ ]:
for var in eval_vars:
    for srname in ["NW", "SE"]:
        IPython.display.display_markdown(f"### {var} {srname}", raw=True)
        rt_ds = xr.concat(
            [
                pred_and_target_return_levels(
                    VAR_DAS[var].cf.sel(**q, method="nearest").sel(time=(VAR_DAS[var]["time"]["time.season"] == season)),
                    var,
                    n_days_per_year=90
                ).expand_dims(location=[f"{season} {label}"])
                for (label, q), season in itertools.product(RANDOM_BOX_LOCATIONS[srname].items(), ["DJF", "JJA"])
            ],
            dim="location",
        ).sortby("location")
    
        plot_return_levels(rt_ds[f"pred_{var}_return_level"], rt_ds[f"target_{var}_return_level"], row="location")
        plt.show()
    
        pretty_return_levels_table(rt_ds, var)

## Return time plots (sorting): subregions (domain max seasonal)

In [ ]:
for var in eval_vars:
    IPython.display.display_markdown(f"### {var}", raw=True)
    rt_ds = xr.concat(
        [
            pred_and_target_return_levels(
                VAR_DAS[var].isel(**SUBREGIONS[srname]).sel(time=(CPM_DAS[var]["time"]["time.season"] == season)).max(dim=["grid_longitude", "grid_latitude"], keep_attrs=True),
                var,
                n_days_per_year=90,
            ).expand_dims(location=[f"{srname} {season}"])
            for (srname, season) in itertools.product(["NW", "SE"], ["DJF", "JJA"])
        ],
        dim="location",
    )
        
    plot_return_levels(rt_ds[f"pred_{var}_return_level"], rt_ds[f"target_{var}_return_level"], row="location")
    plt.show()

    pretty_return_levels_table(rt_ds, var)

## Return time plots (sorting): subregions (domain accumulated seasonal)

In [ ]:
for var in eval_vars:
    IPython.display.display_markdown(f"### {var}", raw=True)
    rt_ds = xr.concat(
        [
            pred_and_target_return_levels(
                VAR_DAS[var].isel(**SUBREGIONS[srname]).sel(time=(CPM_DAS[var]["time"]["time.season"] == season)).sum(dim=["grid_longitude", "grid_latitude"], keep_attrs=True),
                var,
                n_days_per_year=90,
            ).expand_dims(location=[f"{srname} {season}"])
            for (srname, season) in itertools.product(["NW", "SE"], ["DJF", "JJA"])
        ],
        dim="location",
    )
        
    plot_return_levels(rt_ds[f"pred_{var}_return_level"], rt_ds[f"target_{var}_return_level"], row="location")
    plt.show()

    pretty_return_levels_table(rt_ds, var)